In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import re

vectorizer = None  # 定义全局变量

# 下载必要的nltk数据
def download_nltk_data():
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords')


# 文本预处理函数
def preprocess_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    return ' '.join(tokens)


# 构建特征函数
def build_features(train_df, test_df, train_text_cols, test_text_cols):
    global vectorizer  # 声明使用全局变量
    # 应用预处理
    train_df[train_text_cols[0]] = train_df[train_text_cols[0]].apply(preprocess_text)
    train_df[train_text_cols[1]] = train_df[train_text_cols[1]].apply(preprocess_text)
    test_df[test_text_cols[0]] = test_df[test_text_cols[0]].apply(preprocess_text)
    test_df[test_text_cols[1]] = test_df[test_text_cols[1]].apply(preprocess_text)

    # 创建额外特征
    train_df['a_length'] = train_df[train_text_cols[0]].apply(len)
    train_df['b_length'] = train_df[train_text_cols[1]].apply(len)
    test_df['a_length'] = test_df[test_text_cols[0]].apply(len)
    test_df['b_length'] = test_df[test_text_cols[1]].apply(len)

    train_df['a_word_count'] = train_df[train_text_cols[0]].apply(lambda x: len(x.split()))
    train_df['b_word_count'] = train_df[train_text_cols[1]].apply(lambda x: len(x.split()))
    test_df['a_word_count'] = test_df[test_text_cols[0]].apply(lambda x: len(x.split()))
    test_df['b_word_count'] = test_df[test_text_cols[1]].apply(lambda x: len(x.split()))

    # 文本向量化
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    a_vectors = vectorizer.fit_transform(train_df[train_text_cols[0]] + train_df[train_text_cols[1]])
    a_vectors_train = vectorizer.transform(train_df[train_text_cols[0]])
    a_vectors_test = vectorizer.transform(test_df[test_text_cols[0]])

    b_vectors = vectorizer.transform(train_df[train_text_cols[1]])
    b_vectors_test = vectorizer.transform(test_df[test_text_cols[1]])

    # 组合特征
    from scipy.sparse import hstack
    train_features = hstack([
        a_vectors_train,
        b_vectors,
        train_df[['a_length', 'b_length', 'a_word_count', 'b_word_count']].values
    ])
    test_features = hstack([
        a_vectors_test,
        b_vectors_test,
        test_df[['a_length', 'b_length', 'a_word_count', 'b_word_count']].values
    ])
    return train_features, test_features


# 检测目标变量列名并处理
def process_target(train_df):
    target_columns = ['winner_model_a', 'winner_model_b', 'winner_tie']
    y = train_df[target_columns].values.argmax(axis=1)
    le = LabelEncoder()
    y = le.fit_transform(y)
    return y, le


# 主函数
def main():
    # 下载必要的nltk数据
    download_nltk_data()
    # 读取数据
    train_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
    test_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')

    # 自动检测文本列名
    def find_text_columns(df):
        text_columns = []
        for col in df.columns:
            if 'response' in col.lower() or 'answer' in col.lower() or 'text' in col.lower():
                text_columns.append(col)
        return text_columns

    train_text_cols = find_text_columns(train_df)
    test_text_cols = find_text_columns(test_df)

    # 构建特征
    train_features, test_features = build_features(train_df, test_df, train_text_cols, test_text_cols)

    # 处理目标变量
    y, le = process_target(train_df)

    # 划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        train_features, y, test_size=0.2, random_state=42, stratify=y
    )

    # 超参数调优
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 5, 10]
    }
    rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
    grid_search = GridSearchCV(
        estimator=rf_model,
        param_grid=param_grid,
        scoring='neg_log_loss',
        cv=3,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # 验证模型
    val_preds = best_model.predict_proba(X_val)
    logloss = log_loss(y_val, val_preds)
    print("\n==== 模型评估结果 ====")
    print(f"验证集对数损失: {logloss:.4f}")

    # 预测测试集
    test_preds = best_model.predict_proba(test_features)

    # 打印预测结果
    print("\n==== 测试集预测结果样本 ====")
    print("ID\tModel A Prob\tModel B Prob\tTie Prob\tPredicted Winner")
    print("-" * 80)
    for i in range(min(7, len(test_preds))):  # 打印前7个样本
        id_value = test_df['id'].iloc[i]
        probs = test_preds[i]
        # 获取预测的类别
        predicted_class = np.argmax(probs)
        class_names = ['model_a', 'model_b', 'tie']
        predicted_winner = class_names[predicted_class]
        print(f"{id_value}\t{probs[0]:.4f}\t\t{probs[1]:.4f}\t\t{probs[2]:.4f}\t\t{predicted_winner}")

    # 打印总体预测分布
    print("\n==== 总体预测分布 ====")
    predicted_classes = np.argmax(test_preds, axis=1)
    class_counts = np.bincount(predicted_classes)
    for i, count in enumerate(class_counts):
        print(f"{class_names[i]}: {count} ({count/len(predicted_classes):.2%})")

    # 特征重要性分析
    importances = best_model.feature_importances_
    feature_names = np.concatenate([
        np.array(vectorizer.get_feature_names_out()),
        np.array(vectorizer.get_feature_names_out()),
        np.array(['a_length', 'b_length', 'a_word_count', 'b_word_count'])
    ])
    sorted_indices = np.argsort(importances)[::-1]
    print("\n==== 特征重要性 ====")
    for idx in sorted_indices:
        print(f"{feature_names[idx]}: {importances[idx]}")

    # 创建提交文件
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'winner_model_a': test_preds[:, 0],
        'winner_model_b': test_preds[:, 1],
        'winner_tie': test_preds[:, 2]
    })
    
    # 保存提交文件
    submission_df.to_csv('submission.csv', index=False)
    print("\n==== 提交文件已生成 ====")
    print("文件路径: submission.csv")
    print(f"文件内容: {submission_df.shape[0]}行 × {submission_df.shape[1]}列")


if __name__ == "__main__":
    main()